In [1]:
%load_ext autoreload
%autoreload 2
%env ANYWIDGET_HMR=1

env: ANYWIDGET_HMR=1


In [2]:
import pandas as pd
import celldega as dega

In [3]:
dataset = 'Visium_HD_Human_Pancreas_binned_outputs'
base_path = 'data/visium-hd_data/' + dataset + '/binned_outputs/square_008um/'
landscape_files_path = 'data/landscape_files/' + dataset + '/'


In [4]:
pd.read_parquet('data/landscape_files/Visium_HD_Human_Pancreas_binned_outputs/meta_gene.parquet').head()

,mean,std,max,non-zero
SAMD11,0.000294,0.083349,2,0.000002
NOC2L,0.006691,0.396121,2,0.000002
KLHL17,0.001343,0.177495,2,0.000002
PLEKHN1,0.000161,0.061235,1,0.000002
PERM1,0.000076,0.042008,1,0.000002


In [5]:
pd.read_parquet('data/landscape_files/Visium_HD_Human_Pancreas_binned_outputs/meta_tile.parquet').head()

,name,center_x,center_y,cluster
s_008um_00000_00341-1,s_008um_00000_00341-1,3534.410982,1197.435011,5
s_008um_00000_00342-1,s_008um_00000_00342-1,3531.019825,1197.471088,5
s_008um_00000_00415-1,s_008um_00000_00415-1,3283.465021,1200.104710,1
s_008um_00000_00416-1,s_008um_00000_00416-1,3280.073854,1200.140788,4
s_008um_00000_00417-1,s_008um_00000_00417-1,3276.682688,1200.176865,1


# Viz

In [53]:
server_address = dega.viz.get_local_server()

In [54]:
base_url = f'http://localhost:{server_address}/data/landscape_files/' + dataset

In [55]:
from ipywidgets import Widget

In [58]:
Widget.close_all()

In [56]:

landscape_sst = dega.viz.Landscape(
    technology='Visium-HD', 
    base_url=base_url, 
    ini_x=3000, 
    ini_y=4000, 
    ini_z=0, 
    ini_zoom=-3, 
    # square_tile_size=1.25,
    square_tile_size=1.5,
    # width=500
)
landscape_sst

/var/folders/2_/m2fbpk455xg3pbt2h6y9q6980000gn/T/ipykernel_53409/1076125594.py:2: UserWarning: Transformation matrix not found at http://localhost:49377/data/landscape_files/Visium_HD_Human_Pancreas_binned_outputs/micron_to_image_transform.csv. Using identity.
  landscape_sst = dega.viz.Landscape(


Landscape(base_url='http://localhost:49377/data/landscape_files/Visium_HD_Human_Pancreas_binned_outputs', cell…

In [6]:
import scanpy as sc

In [7]:
adata = sc.read_10x_mtx('data/visium-hd_data/Visium_HD_Human_Pancreas_binned_outputs/binned_outputs/square_008um/raw_feature_bc_matrix/')

In [8]:
adata

AnnData object with n_obs × n_vars = 702244 × 39125
    var: 'gene_ids', 'feature_types'

In [9]:
adata.obs.head()

""
s_008um_00000_00000-1
s_008um_00000_00001-1
s_008um_00000_00002-1
s_008um_00000_00003-1
s_008um_00000_00004-1


In [10]:
meta_tile = pd.read_parquet('data/landscape_files/Visium_HD_Human_Pancreas_binned_outputs/meta_tile.parquet')
meta_tile.head()

,name,center_x,center_y,cluster
s_008um_00000_00341-1,s_008um_00000_00341-1,3534.410982,1197.435011,5
s_008um_00000_00342-1,s_008um_00000_00342-1,3531.019825,1197.471088,5
s_008um_00000_00415-1,s_008um_00000_00415-1,3283.465021,1200.104710,1
s_008um_00000_00416-1,s_008um_00000_00416-1,3280.073854,1200.140788,4
s_008um_00000_00417-1,s_008um_00000_00417-1,3276.682688,1200.176865,1


In [11]:
adata.obs['cluster'] = meta_tile['cluster']

In [12]:
# !pip install --user scikit-misc

In [18]:
sc.pp.calculate_qc_metrics(adata, inplace=True)

In [26]:
keep_genes = adata.var['mean_counts'].sort_values(ascending=False).index.tolist()[:5000]

# Subset AnnData using .var_names (gene names must be the index of adata.var)
adata = adata[:, adata.var_names.isin(keep_genes)]


In [32]:
# Step 1: Create a boolean mask for non-NaN cluster entries
mask = ~adata.obs['cluster'].isna()

# Step 2: Subset AnnData using the mask (filters rows = cells)
adata = adata[mask, :]

In [38]:
import numpy as np

In [39]:
# Step 1: Extract expression matrix and cluster labels
X = adata.X  # This may be sparse
clusters = adata.obs['cluster']

# Step 2: Convert sparse matrix to dense
# If memory is tight, do this in chunks — but here we'll assume it fits
X_dense = X.toarray() if not isinstance(X, np.ndarray) else X

# Step 3: Create DataFrame for easier groupby
df = pd.DataFrame(X_dense, index=adata.obs_names, columns=adata.var_names)
df['cluster'] = clusters.values

# Step 4: Group by cluster and compute mean
cluster_means = df.groupby('cluster').mean().drop(columns='cluster', errors='ignore')

In [41]:
df_sig = cluster_means.T
df_sig

cluster,1,10,11,12,13,14,2,3,4,5,6,7,8,9
NOC2L,0.009191,0.012044,0.008427,0.021016,0.002392,0.007557,0.000229,0.009996,0.006658,0.001607,0.002941,0.013553,0.012040,0.010909
HES4,0.016765,0.007527,0.012172,0.017513,0.007177,0.047859,0.000510,0.050016,0.006909,0.002552,0.007058,0.014268,0.013218,0.012867
AGRN,0.025407,0.016560,0.023408,0.054291,0.004785,0.020151,0.000752,0.085474,0.010981,0.004442,0.007231,0.021157,0.022379,0.028531
SDF4,0.062657,0.048175,0.038858,0.066550,0.007177,0.017632,0.001160,0.057972,0.054518,0.008727,0.012213,0.058013,0.098940,0.039441
UBE2J2,0.010968,0.010162,0.012172,0.011384,0.007177,0.010076,0.000217,0.012165,0.008282,0.001386,0.002145,0.011943,0.012826,0.009231
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
DEPRECATED_SUPT4H1,0.015779,0.027475,0.020599,0.028021,0.009569,0.017632,0.000306,0.020381,0.011988,0.002772,0.005536,0.011764,0.021332,0.012587
DEPRECATED_VAMP2,0.010600,0.040271,0.026217,0.011384,0.016746,0.108312,0.000242,0.013742,0.006863,0.001701,0.003321,0.010690,0.012695,0.014825
DEPRECATED_PDCD6,0.012884,0.022582,0.013109,0.018389,0.000000,0.012594,0.000255,0.015466,0.009220,0.002300,0.003806,0.016102,0.016883,0.012587
DEPRECATED_RBM8A,0.016785,0.031615,0.024345,0.031524,0.009569,0.035264,0.000395,0.021809,0.012812,0.003024,0.005639,0.016594,0.020154,0.017622


In [51]:
mat = dega.clust.Matrix(data=df_sig)
# mat.norm(by='total', axis='col')
mat.norm(by='zscore', axis='row')
mat.cluster()
cgm = dega.viz.Clustergram(matrix=mat)

In [52]:
cgm

Clustergram(network_meta={'linkage': {}, 'row_attr': [], 'col_attr': [], 'row_attr_maxabs': [], 'col_attr_maxa…

In [57]:
dega.viz.landscape_clustergram(landscape=landscape_sst, mat=cgm)

AttributeError: 'NoneType' object has no attribute 'comm_id'